# CNDE NDT Document QA — Grounded RAG with Automated Verification
Run the sections **in order** on a fresh Colab runtime with a **T4 GPU** (`Runtime > Change runtime type`).
Reusable logic lives in `src/`; this notebook only wires it together.

## 1. Environment setup

In [43]:
import sys, platform, shutil, subprocess
print("Python", sys.version.split()[0], "|", platform.platform())
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    print("NO GPU on this runtime -> switch to a T4 GPU runtime before section 5.")

Python 3.13.16 | Linux-6.6.159+-x86_64-with-glibc2.39
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07



## 2. Dependency installation
Uses the code synced by the Colab folder-sync extension (`/content/cnde`) if present; otherwise clones the public repo. torch is preinstalled on Colab and is not reinstalled.

In [44]:
import os
REPO_URL = "https://github.com/AbhijeetKumar1505/CNDE.git"
BRANCH = "main"                # or a feature branch, e.g. "feature/llm-verification"
SYNCED_DIR = "/content/cnde"    # local folder synced by the Colab sync extension
CLONE_DIR = "/content/CNDE"

if os.path.isdir(os.path.join(SYNCED_DIR, "src")):
    REPO_DIR = SYNCED_DIR
    print("Using synced working copy:", REPO_DIR)
else:
    REPO_DIR = CLONE_DIR
    if not os.path.isdir(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}
    !git -C {REPO_DIR} log --oneline -3
%cd {REPO_DIR}
%pip install -q -r requirements.txt

remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 17 (delta 11), reused 9 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (17/17), 28.04 KiB | 1.65 MiB/s, done.
From https://github.com/AbhijeetKumar1505/CNDE
   1f0d2bf..b815767  main              -> origin/main
   49447f8..98ccd2b  feature/llm-verification -> origin/feature/llm-verification
   f6e2a24..f211f31  feature/retrieval -> origin/feature/retrieval
Already on 'main'
Your branch is behind 'origin/main' by 5 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/AbhijeetKumar1505/CNDE
 * branch            main       -> FETCH_HEAD
Updating 1f0d2bf..b815767
Fast-forward
 main.ipynb          | 6338 +++++++++++++++++++++++++++++++++++++++++++++++----
 src/generation.py   |    1 +
 src/ingestion.py    |   64 +-
 src/verification.py |    7 +
 4 files changed, 6000 insertions(+),

## 3. Google Drive mounting
Persistent storage: `MyDrive/CNDE_Hackathon/`. If mounting is unavailable (e.g. VS Code Colab kernel), `Config` falls back to `<repo>/workspace` on the runtime, which does not persist across runtimes.

In [45]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:  # report, don't hide: storage falls back to the runtime-local workspace
    print(f"Drive NOT mounted ({type(e).__name__}: {e}). Using runtime-local workspace instead.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Configuration

In [46]:
import sys, importlib
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:  # IPython 7.x autoreload imports `imp`, which Python 3.13 removed
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError as e:
    print(f"autoreload unavailable ({e}). After code changes run reload_src(), then re-run the cell you need.")

def reload_src():
    """Reload all project modules (fallback when autoreload is unavailable)."""
    import src
    for name in sorted(n for n in sys.modules if n.startswith("src.")):
        importlib.reload(sys.modules[name])
    print("reloaded:", sorted(n for n in sys.modules if n.startswith("src.")))

from src.config import Config
cfg = Config()
cfg.make_dirs()
cfg

autoreload unavailable (No module named 'imp'). After code changes run reload_src(), then re-run the cell you need.


Config(root=PosixPath('/content/drive/MyDrive/CNDE_Hackathon'), llm_id='Qwen/Qwen3-4B-Instruct-2507', llm_fallback_id='Qwen/Qwen2.5-1.5B-Instruct', embed_id='BAAI/bge-small-en-v1.5', embed_device='auto', chunk_tokens=350, chunk_overlap=50, top_k=5, min_retrieval_score=0.0, max_input_tokens=4096, max_new_tokens=512)

## 5. Model loading
Qwen3-4B-Instruct-2507 in 4-bit NF4 (double quant, FP16 compute). The smoke test checks for FP16 overflow (non-finite logits) and records VRAM and decode speed.

In [47]:
from src.generation import load_llm, smoke_test
model, tokenizer = load_llm(cfg.llm_id)
smoke = smoke_test(model, tokenizer)
smoke

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B-Instruct-2507 in 37.0s | {'vram_allocated_gb': 8.18, 'vram_peak_gb': 8.22, 'vram_device_used_gb': 8.38, 'vram_total_gb': 15.64}


{'text': 'Ultrasonic testing in non-destructive testing (NDT) is a method that uses high-frequency sound waves to detect flaws, measure thickness, or assess material properties without damaging the component.',
 'logits_finite': True,
 'new_tokens': 39,
 'tokens_per_s': 11.7,
 'vram_allocated_gb': 7.4,
 'vram_peak_gb': 8.22,
 'vram_device_used_gb': 8.39,
 'vram_total_gb': 15.64}

## 6. Document upload

In [48]:
import requests, pymupdf
from pathlib import Path

# Public-domain corpus. FAA publishes AC 43.13-1B only as one 646-page PDF; Chapter 5 (NDI) is physical
# pages 222-271 and is extracted below, so citations refer to pages of the extracted file.
NASA_URL = "https://standards.nasa.gov/sites/default/files/standards/NASA/C/0/2023-08-03-NASA-STD-5009C-Approved.pdf"
FAA_URL = "https://www.faa.gov/documentLibrary/media/Advisory_Circular/AC_43.13-1B_w-chg1.pdf"

def download(url, dest):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
    r.raise_for_status()
    Path(dest).write_bytes(r.content)

nasa, faa_ch5 = cfg.docs_dir / "NASA-STD-5009C.pdf", cfg.docs_dir / "FAA_AC_43.13-1B_Ch5_NDI.pdf"
if not nasa.exists():
    download(NASA_URL, nasa)
if not faa_ch5.exists():
    full = cfg.root / "documents" / "source" / "AC_43.13-1B_w-chg1.pdf"   # full AC kept outside docs_dir
    full.parent.mkdir(parents=True, exist_ok=True)
    if not full.exists():
        download(FAA_URL, full)
    src, out = pymupdf.open(full), pymupdf.open()
    out.insert_pdf(src, from_page=221, to_page=270)   # 0-based -> physical pages 222..271
    out.save(str(faa_ch5))

# Add your own PDF/DOCX/TXT files to this folder (or upload them in the Gradio app).
doc_paths = sorted(p for p in cfg.docs_dir.iterdir() if p.suffix.lower() in {".pdf", ".docx", ".txt"})
for p in doc_paths:
    print(f"{p.name:40s} {p.stat().st_size / 1e6:6.2f} MB")

FAA_AC_43.13-1B_Ch5_NDI.pdf                1.35 MB
NASA-STD-5009C.pdf                         0.77 MB


## 7. Text extraction

In [49]:
import time
from src.ingestion import extract_document

extracted = {}
for p in doc_paths:
    t0 = time.perf_counter()
    extracted[p.name] = pages = extract_document(p)
    empty = [pg["page"] for pg in pages if not pg["blocks"]]
    print(f"{p.name}: {len(pages)} pages, {sum(len(pg['blocks']) for pg in pages)} blocks, "
          f"pages without usable text {empty} ({time.perf_counter() - t0:.2f}s)")

sample = extracted[doc_paths[0].name][0]
print(f"\nSample: page {sample['page']} of {sample['source']}")
for b in sample["blocks"][:6]:
    print(" -", (f"[heading: {b['heading']}] " if b["heading"] else "") + b["text"][:150])

FAA_AC_43.13-1B_Ch5_NDI.pdf: 50 pages, 586 blocks, pages without usable text [] (0.18s)
NASA-STD-5009C.pdf: 54 pages, 493 blocks, pages without usable text [5, 7] (0.19s)

Sample: page 1 of FAA_AC_43.13-1B_Ch5_NDI.pdf
 - [heading: CHAPTER 5.  NONDESTRUCTIVE INSPECTION (NDI)] CHAPTER 5. NONDESTRUCTIVE INSPECTION (NDI)
 - [heading: SECTION 1. GENERAL] SECTION 1. GENERAL
 - techniques, and practices acceptable to the administrator. If the maintenance instructions include materials, parts, tools, equipment, or test apparat
 - [heading: 5-3. NDT LEVELS.] 5-3. NDT LEVELS. Reference Air Transport Association (ATA) Specification 105- Guidelines For Training and Qualifying Personnel In Nondestructive Testi
 - a. Level I Special.
 - [heading: 5-1. GENERAL.] 5-1. GENERAL. The field of NDI is too varied to be covered in detail in this Advisory Circular (AC). This chapter provides a brief description of the 


## 8. Chunking

In [50]:
import statistics
from src.ingestion import chunk_pages
from src.retrieval import load_embedder, token_counter

embedder = load_embedder(cfg.embed_id, cfg.embed_device)   # its tokenizer defines the chunk budget
count_tokens = token_counter(embedder)
t0 = time.perf_counter()
chunks = [c for pages in extracted.values()
          for c in chunk_pages(pages, count_tokens, cfg.chunk_tokens, cfg.chunk_overlap)]
assert len({c["chunk_id"] for c in chunks}) == len(chunks), "duplicate chunk ids"
n = [c["n_tokens"] for c in chunks]
print(f"{len(chunks)} chunks in {time.perf_counter() - t0:.1f}s | BGE tokens min/median/max "
      f"{min(n)}/{statistics.median(n)}/{max(n)} | embedder on {embedder.device}")
chunks[len(chunks) // 2]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (515 > 512). Running this sequence through the model will result in indexing errors


190 chunks in 0.4s | BGE tokens min/median/max 8/268.5/357 | embedder on cuda:0


{'chunk_id': 'FAA_AC_43_13_1B_Ch5_NDI_pdf-p49-c0095',
 'source': 'FAA_AC_43.13-1B_Ch5_NDI.pdf',
 'page': 49,
 'section': 'SECTION 10. THERMOGRAPHY',
 'text': 'SECTION 10. THERMOGRAPHY\nheated or cooled. The greater the material’s resistance to heat flow, the more readily the flow can be identified due to temperature differences caused by the flaw.\n5-121.–5-126. [RESERVED.]\n5-120. GENERAL. Thermography is an NDI technique that uses radiant electromagnetic thermal energy to detect flaws. The presence of a flaw is indicated by an abnormal temperature variant when the item is subjected to normal heating and cooling conditions inherent to the in-service life, and/or when artificially\nSECTION 11. HOLOGRAPHY\non film, or other electronic recording medium, creating a double image. Indications of applied stresses or defects are shown as virtual images with a system of fringe lines overlaying the part. Holography is most commonly used for rapid assessment of surface flaws in composite structu

## 9. Embedding generation

In [51]:
from src.retrieval import build_index

t0 = time.perf_counter()
index = build_index(chunks, embedder)   # normalized BGE vectors -> FAISS inner product == cosine
print(f"Embedded {index.ntotal} chunks ({index.d}-d) in {time.perf_counter() - t0:.1f}s on {embedder.device}")

Embedded 190 chunks (384-d) in 1.3s on cuda:0


## 10. FAISS indexing

In [52]:
from src.retrieval import load_index, retrieve, save_index

save_index(index, chunks, cfg.index_dir,
           meta={"embed_id": cfg.embed_id, "chunk_tokens": cfg.chunk_tokens, "chunk_overlap": cfg.chunk_overlap})
index, chunks = load_index(cfg.index_dir, expected_embed_id=cfg.embed_id)   # round-trip check
print(f"Saved + reloaded: {index.ntotal} vectors == {len(chunks)} chunks -> {cfg.index_dir}")

for q in ["What should be done with magnetic particles after a magnetic particle inspection?",
          "What crack sizes can standard penetrant NDE reliably detect?",
          "What is the boiling point of mercury?"]:
    t0 = time.perf_counter()
    hits = retrieve(q, index, chunks, embedder, cfg.top_k)
    print(f"\nQ: {q}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")
    for h in hits[:3]:
        print(f"  {h['score']:.3f} | {h['source']} p.{h['page']} | {h['section'][:60]}")

Saved + reloaded: 190 vectors == 190 chunks -> /content/drive/MyDrive/CNDE_Hackathon/indexes

Q: What should be done with magnetic particles after a magnetic particle inspection?  (23 ms)
  0.828 | FAA_AC_43.13-1B_Ch5_NDI.pdf p.26 | 5-51. DEMAGNETIZATION AND  POST- EXAMINATION CLEANING.
  0.814 | FAA_AC_43.13-1B_Ch5_NDI.pdf p.17 | SECTION 4. MAGNETIC PARTICLE INSPECTION
  0.805 | FAA_AC_43.13-1B_Ch5_NDI.pdf p.20 | 5-43. ELECTRICAL MAGNETIZING EQUIPMENT.

Q: What crack sizes can standard penetrant NDE reliably detect?  (14 ms)
  0.820 | NASA-STD-5009C.pdf p.16 | 4.1.1 Standard NDE Crack Sizes
  0.783 | NASA-STD-5009C.pdf p.25 | 4.2 Special NDE
  0.782 | NASA-STD-5009C.pdf p.37 | 4. NDE Requirements

Q: What is the boiling point of mercury?  (13 ms)
  0.552 | NASA-STD-5009C.pdf p.19 | 4.1.4.1 Eddy Current
  0.548 | FAA_AC_43.13-1B_Ch5_NDI.pdf p.3 | 5-5. FLAWS.
  0.548 | NASA-STD-5009C.pdf p.41 | 4.1.4.1.1 Eddy Current


## 11. Question answering

In [53]:
from src.generation import gpu_memory
from src.pipeline import answer_question

DEMO_QUESTIONS = [
    "What should be done with magnetic particles after a magnetic particle inspection?",
    "What crack sizes can standard penetrant NDE reliably detect?",
    "What is the boiling point of mercury?",   # out of scope: expect abstention
]
results = []
for q in DEMO_QUESTIONS:
    r = answer_question(q, index, chunks, embedder, model, tokenizer, cfg, variant="v2")
    results.append(r)
    print(f"Q: {q}\n  retrieval {r['retrieval_ms']} ms | generation {r['latency_s']} s | prompt {r['prompt_tokens']} tok"
          f" | new {r['new_tokens']} tok | gated={r['gated']} | hit_max={r['hit_max_tokens']}")
    print("  RAW:", r["answer"][:1000], "\n")
print(gpu_memory())

Q: What should be done with magnetic particles after a magnetic particle inspection?
  retrieval 15.9 ms | generation 13.44 s | prompt 2146 tok | new 154 tok | gated=False | hit_max=False
  RAW: Answer: After a magnetic particle inspection, the magnetic particles should be thoroughly removed by rinsing the part with a detergent-base cleaner until all magnetic particles are removed, followed by rinsing in a solution of water and rust inhibitor when using water suspension methods [S1][S2]. For cadmium-plated parts, an air-water vapor blast may be used to remove any remaining magnetic particle residue [S1]. Additionally, after final cleaning and drying, temporary protective coatings should be applied when necessary to prevent corrosion [S1].

Limitations: The evidence does not specify the exact method or duration for removing magnetic particles from non-cadmium-plated parts or detail the process for dry methods beyond stating they are not recommended for aerospace components.

Conflicts: 

## 12. Evidence verification

In [54]:
from IPython.display import Markdown, display
from src.pipeline import to_markdown

for q, r in zip(DEMO_QUESTIONS, results):
    display(Markdown(f"---\n## Q: {q}\n" + to_markdown(r)))
    for c in r["verification"]["claims"]:
        print(f"  claim ok={c['supported']} sem={c['semantic']} lex={c['lexical']} cited={c['cited']} | {c['text'][:90]}")

---
## Q: What should be done with magnetic particles after a magnetic particle inspection?
### Answer
After a magnetic particle inspection, the magnetic particles should be thoroughly removed by rinsing the part with a detergent-base cleaner until all magnetic particles are removed, followed by rinsing in a solution of water and rust inhibitor when using water suspension methods [S1][S2]. For cadmium-plated parts, an air-water vapor blast may be used to remove any remaining magnetic particle residue [S1]. Additionally, after final cleaning and drying, temporary protective coatings should be applied when necessary to prevent corrosion [S1].

### Supporting evidence
**[S1]** FAA_AC_43.13-1B_Ch5_NDI.pdf, p. 26 — 5-51. DEMAGNETIZATION AND  POST- EXAMINATION CLEANING. (retrieval score 0.83)
> (5) After magnetic particle examination has been completed, restore any removed finishes according to the manufacturer’s repair manual.
> (2) When parts or materials have been examined using water suspension methods, completely remove the water by any suitable means, such as an air blast, to ensure that the parts are dried immediately after cleaning. Thoroughly rinse the part with a detergentbase cleaner until all magnetic particles are removed. Then rinse in a solution of water and rust inhibitor.
> (3) For cadmium-plated parts an airwater vapor blast may be used to remove any remaining magnetic particle residue.
> NOTE: Visible penetrant is often used interchangeably by NDI personnel with fluore …

**[S2]** FAA_AC_43.13-1B_Ch5_NDI.pdf, p. 17 — SECTION 4. MAGNETIC PARTICLE INSPECTION (retrieval score 0.81)
> SECTION 4. MAGNETIC PARTICLE INSPECTION
> FIGURE 5-10. Crack detection by magnetic particle inspection.
> 5-40. GENERAL. Magnetic particle inspection is a method for detecting cracks, laps, seams, voids, pits, subsurface holes, and other surface, or slightly subsurface, discontinuities in ferro-magnetic materials. Magnetic particle inspection can be used only on ferro-magnetic materials (iron and steel). It can be performed on raw material, billets, finished and semifinished materials, welds, and in-service assembled or disassembled parts. Magnetic particles are applied over a surface either dry, as a powder, or wet, as particles in a liquid carrier such as oil or water.
> Common uses for magnetic …

### References
- [S1] FAA_AC_43.13-1B_Ch5_NDI.pdf, p. 26 — 5-51. DEMAGNETIZATION AND  POST- EXAMINATION CLEANING.
- [S2] FAA_AC_43.13-1B_Ch5_NDI.pdf, p. 17 — SECTION 4. MAGNETIC PARTICLE INSPECTION

### Verification
**Status: Supported** (rule-based checks, not a probability)
citation validity 1.00 · numeric fidelity 1.00 · claim support 1.00 · grounding score 1.00

### Limitations
The evidence does not specify the exact method or duration for removing magnetic particles from non-cadmium-plated parts or detail the process for dry methods beyond stating they are not recommended for aerospace components.

  claim ok=True sem=0.829 lex=0.842 cited=['S1', 'S2'] | After a magnetic particle inspection, the magnetic particles should be thoroughly removed 
  claim ok=True sem=0.974 lex=0.9 cited=['S1'] | For cadmium-plated parts, an air-water vapor blast may be used to remove any remaining mag
  claim ok=True sem=0.778 lex=0.833 cited=['S1'] | Additionally, after final cleaning and drying, temporary protective coatings should be app


---
## Q: What crack sizes can standard penetrant NDE reliably detect?
### Answer
Standard penetrant NDE can reliably detect cracks with dimensions as follows:
- Open surface, 0.64 mm ≤ t ≤ 1.27 mm (thickness), crack dimension a* = 0.64 mm, crack dimension c* = 1.91 mm;
- Through corner, t > 1.91 mm, crack dimension a* = 2.54 mm, crack dimension c* = 3.81 mm;
- Edge or hole, 0.64 mm ≤ t ≤ 2.54 mm, crack dimension a* = 0.64 mm, crack dimension c* = 3.81 mm;
- Edge or hole, t > 2.54 mm, crack dimension a* = 3.81 mm, crack dimension c* = 3.81 mm.
[S3][S4]

*Interpretation (not a documented fact):* The values are consistent across S3 and S4, with minor formatting differences but identical numerical values and conditions.

### Supporting evidence
**[S3]** NASA-STD-5009C.pdf, p. 37 — 4. NDE Requirements (retrieval score 0.78)
> Table 1—Reliably Detectable Flaw Sizes for Fracture Analysis Based on Standard NDE Methods (Millimeters) (See Section
> 4.1 of this Standard for applicability.)
> Système Internationale Units (millimeters)
> Crack
> Crack Location
> Part Thickness, t
> Type
> Crack Dimension, a*
> Crack Dimension, c*
> Eddy Current NDE
> Open Surface 0.381≤ t ≤ 1.27 Through t 1.27
> Through
> Corner
> t > 1.27 PTC1 0.51 1.27 2.54 1.27 Edge or Hole 0.381≤ t ≤ 1.91 t > 1.91
> t 1.91 2.54 1.91 Penetrant NDE
> Through Through
> PTC
> Open Surface 0.64 ≤ t ≤ 1.27 1.27<t <1.91 t > 1.91
> t t 0.64 1.91
> Through
> Corner
> 2.54 3.81 – t 3.18 1.91 Edge or Hole 0.64 ≤ t ≤ 2.54 t > 2.54
> t 2.54 3.81 3.81 Magnetic Particle NDE
> Through
> PTC
> Open Surface t ≤ 1.91  …

**[S4]** NASA-STD-5009C.pdf, p. 33 — 4.4.3 Supporting Data and Record Retention (retrieval score 0.77)
> Table 1—Reliably Detectable Flaw Sizes for Fracture Analysis Based on Standard NDE
> Methods (Millimeters) (See Section 4.1 of this Standard for applicability.)
> Système Internationale Units (millimeters)
> Crack
> Crack Location
> Part Thickness, t
> Type
> Crack Dimension, a*
> Crack Dimension, c*
> Eddy Current NDE
> Open Surface 0.381 ≤ t ≤ 1.27 Through t 1.27
> Through
> Corner
> t 1.91 2.54 1.91
> t > 1.27 PTC1 0.51 1.27 2.54 1.27 Edge or Hole 0.381 ≤ t ≤ 1.91 t > 1.91
> Penetrant NDE
> Through Through
> PTC
> Open Surface 0.64 ≤ t ≤ 1.27 1.27<t <1.91 t > 1.91
> t t 0.64 1.91
> Through
> Corner
> t 2.54 3.81 3.81
> 2.54 3.81 – t 3.18 1.91 Edge or Hole 0.64 ≤ t ≤ 2.54 t > 2.54
> Magnetic Particle NDE
> Through
> PTC
> Open Surface t ≤ 1.9 …

### References
- [S3] NASA-STD-5009C.pdf, p. 37 — 4. NDE Requirements
- [S4] NASA-STD-5009C.pdf, p. 33 — 4.4.3 Supporting Data and Record Retention

### Verification
**Status: Supported** (rule-based checks, not a probability)
citation validity 1.00 · numeric fidelity 1.00 · claim support 1.00 · grounding score 1.00

### Limitations
The evidence does not specify the part thickness range or crack location for all cases in a unified format, and the values are presented in tables with conditional applicability based on thickness and location.

  claim ok=True sem=0.868 lex=0.765 cited=['S3', 'S4'] | Standard penetrant NDE can reliably detect cracks with dimensions as follows:
- Open surfa


---
## Q: What is the boiling point of mercury?
### Answer
INSUFFICIENT_EVIDENCE: The evidence provided does not contain any information about the boiling point of mercury.

### Verification
**Status: Insufficient evidence** (rule-based checks, not a probability)

## 13. Evaluation

In [55]:
# TODO: Milestone 5

## 14. Demo execution

In [56]:
# TODO: Milestone 5 — Gradio app